In [7]:
### Import Dependencies and Set Reproducibility Controls
import numpy as np
import pandas as pd
from pathlib import Path

SEED = 42
np.random.seed(SEED)

In [15]:
### Configure Local Data Paths (GDPR-Compliant)
project_root = Path("..").resolve()
raw_csv_path = project_root / "data" / "raw_batch1" / "emails.csv"
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Raw data path:", raw_csv_path)
print("Processed dir:", processed_dir)

Project root: C:\Users\willd\Dissertation
Raw data path: C:\Users\willd\Dissertation\data\raw_batch1\emails.csv
Processed dir: C:\Users\willd\Dissertation\data\processed


In [17]:
### Load Raw Dataset
df = pd.read_csv(raw_csv_path)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head(3)

Shape: (185, 2)
Columns: ['subject', 'body']


,subject,body
0,Module Registration Issue,"Dear Dr Smith,\r\n\r\nI'm trying to register f..."
1,changing modules,hi\r\n\r\ni want to drop kv4006 and switch to ...
2,Student ID Card Issue,"Dear Dr Smith,\r\n\r\nApologies for emailing y..."


In [18]:
### Validate Dataset Schema and Data Quality
required_cols = ["subject", "body"]
missing_cols = [c for c in required_cols if c not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

print({
    "rows": len(df),
    "missing_subject": int(df["subject"].isna().sum()),
    "missing_body": int(df["body"].isna().sum()),
})

{'rows': 185, 'missing_subject': 0, 'missing_body': 0}


In [11]:
### Define PII Masking and Text Cleaning Helpers
import re

PII_PATTERNS = {
    "EMAIL": r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b",
    "PHONE": r"\b(?:\+44\s?7\d{3}|07\d{3})\s?\d{3}\s?\d{3}\b",
    "STUDENT_ID": r"\b(?:w|W)?\d{7,9}\b",
}

SIGN_OFF_PATTERNS = [
    r"(?im)^kind regards,?.*$",
    r"(?im)^regards,?.*$",
    r"(?im)^thanks,?.*$",
    r"(?im)^many thanks,?.*$",
]

def mask_pii(text: str) -> str:
    if not isinstance(text, str):
        text = "" if pd.isna(text) else str(text)
    masked = text
    for label, pattern in PII_PATTERNS.items():
        masked = re.sub(pattern, f"[{label}]", masked)
    return masked

def clean_email_text(text: str) -> str:
    if not isinstance(text, str):
        text = "" if pd.isna(text) else str(text)

    cleaned = text
    cleaned = re.sub(r"(?is)\n?On .*? wrote:.*$", " ", cleaned)
    cleaned = re.sub(r"(?im)^>.*$", " ", cleaned)
    cleaned = re.sub(r"https?://\S+|www\.\S+", " [URL] ", cleaned)

    for p in SIGN_OFF_PATTERNS:
        cleaned = re.sub(p, " ", cleaned)

    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned

In [12]:
### Apply PII Masking and Text Cleaning
if "df" not in globals():
    df = pd.read_csv(raw_csv_path)

df_email = df.copy()
df_email["subject_masked"] = df_email["subject"].apply(mask_pii)
df_email["body_masked"] = df_email["body"].apply(mask_pii)
df_email["subject_clean"] = df_email["subject_masked"].apply(clean_email_text)
df_email["body_clean"] = df_email["body_masked"].apply(clean_email_text)
df_email["text_clean"] = (df_email["subject_clean"] + " [SEP] " + df_email["body_clean"]).str.strip()

In [19]:
### Inspect Transformation Results
sample_idx = 0
print("ORIGINAL SUBJECT:")
print(df_email.loc[sample_idx, "subject"])
print("\nCLEANED SUBJECT:")
print(df_email.loc[sample_idx, "subject_clean"])
print("\nORIGINAL BODY:")
print(df_email.loc[sample_idx, "body"])
print("\nCLEANED BODY:")
print(df_email.loc[sample_idx, "body_clean"])

ORIGINAL SUBJECT:
Module Registration Issue

CLEANED SUBJECT:
Module Registration Issue

ORIGINAL BODY:
Dear Dr Smith,

I'm trying to register for KV6003 on the student portal but the system says the module is full. I need this module to complete my degree this year. Could you advise on whether additional places will be made available or if there's someone I should contact to resolve this?

Thank you,
Sarah Thompson

CLEANED BODY:
Dear Dr Smith, I'm trying to register for KV6003 on the student portal but the system says the module is full. I need this module to complete my degree this year. Could you advise on whether additional places will be made available or if there's someone I should contact to resolve this? Thank you, Sarah Thompson


In [14]:
### Save Preprocessing Artifact
email_path = processed_dir / "email_masked_cleaned.csv"
df_email.to_csv(email_path, index=False)
print(email_path)

C:\Users\willd\Dissertation\data\processed\email_masked_cleaned.csv
